In [21]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
from tqdm import tqdm

# Load base tables
uscities = pd.read_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/uscities.csv')
auto_make_model = pd.read_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_MAKE_MODEL.csv')
auto_insurance_agent_master = pd.read_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_INSURANCE_AGENT_MASTER.csv')
auto_insurance_policy_master = pd.read_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_INSURANCE_POLICY_MASTER.csv')
auto_insurance_claims_master = pd.read_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_INSURANCE_CLAIMS_MASTER.csv')

# Initialize the synthetic data DataFrame
synthetic_data = pd.DataFrame()

# Define the base year and number of records for the base year
base_year = 2015

# Define the date range for application dates
start_date = datetime.strptime('2006-01-01', '%Y-%m-%d')
end_date = datetime.strptime('2024-12-31', '%Y-%m-%d')

# Generate synthetic data for each year from base year to 2024
for year in tqdm(range(base_year, 2025), desc="Generating data"):
    # Get the number of records for the current year from AUTO_INSURANCE_POLICY_MASTER
    policy_records = auto_insurance_policy_master[auto_insurance_policy_master['POLICY_BIND_DATE'].str.startswith(str(year))].shape[0]
    
    # Calculate the total number of records for the current year
    total_records = int(policy_records / 0.37)
    additional_records = total_records - policy_records
    
    # Generate application IDs
    application_ids = [f'APP{str(i).zfill(8)}' for i in range(len(synthetic_data), len(synthetic_data) + total_records)]
    
    # Generate application dates
    application_dates = [start_date + timedelta(days=random.randint(0, (end_date - start_date).days)) for _ in range(total_records)]
    
    # Ensure APPLICATION_DATE is always lesser than POLICY_BIND_DATE
    application_dates = [date if date < datetime.strptime(bind_date, '%Y-%m-%d') else datetime.strptime(bind_date, '%Y-%m-%d') - timedelta(days=random.randint(1, 10)) for date, bind_date in zip(application_dates, auto_insurance_policy_master['POLICY_BIND_DATE'].sample(total_records, replace=True))]
    
    # Generate quote converted status
    quote_converted = ['Quote Converted'] * policy_records + ['Quote not converted'] * additional_records
    
    # Generate policy numbers, customer IDs, policy premiums, deductibles, CSLs, bind dates, agent IDs
    policy_numbers = list(auto_insurance_policy_master['POLICY_NUMBER'].sample(policy_records)) + [None] * additional_records
    customer_ids = list(auto_insurance_policy_master['CUSTOMER_ID'].sample(policy_records)) + [None] * additional_records
    policy_premiums = list(auto_insurance_policy_master['POLICY_PREMIUM'].sample(policy_records)) + [None] * additional_records
    policy_deductibles = list(auto_insurance_policy_master['POLICY_DEDUCTIBLE'].sample(policy_records)) + [None] * additional_records
    policy_csls = list(auto_insurance_policy_master['POLICY_CSL'].sample(policy_records)) + [None] * additional_records
    policy_bind_dates = list(auto_insurance_policy_master['POLICY_BIND_DATE'].sample(policy_records)) + [None] * additional_records
    agent_ids = list(auto_insurance_policy_master['AGENT_ID'].sample(policy_records)) + [None] * additional_records
    
    # Generate prospect IDs
    prospect_ids = [f'PR{str(i).zfill(8)}' for i in range(len(synthetic_data), len(synthetic_data) + total_records)]
    
    # Generate prospect state, county, and city
    prospect_state = ['Texas'] * total_records
    prospect_county_city = uscities[uscities['STATE'] == 'Texas'][['COUNTY', 'CITY']]
    prospect_counties_cities = list(prospect_county_city.sample(total_records, replace=True).itertuples(index=False, name=None))
    
    # Separate counties and cities into two lists
    prospect_counties, prospect_cities = zip(*prospect_counties_cities)
    
    # Generate auto make and model
    auto_makes_models = list(auto_make_model.sample(total_records, replace=True).itertuples(index=False, name=None))
    
    # Separate auto makes and models into two lists
    auto_makes, auto_models = zip(*auto_makes_models)
    
    # Generate expected CSLs
    expected_csls = [csl if csl is not None else random.choices(['100/300', '250/500', '500/1000'], weights=[0.47, 0.38, 0.15])[0] for csl in policy_csls]
    
    # Generate deductible looked for
    deductible_looked_for = [deductible if deductible is not None else random.choices([250, 500, 1000, 2000], weights=[0.15, 0.45, 0.30, 0.10])[0] for deductible in policy_deductibles]
    
    # Generate expected premiums with updated logic
    expected_premiums = []
    for premium in policy_premiums:
        if premium is not None:
            expected_premiums.append(premium * (1 + random.uniform(-0.3, 0.3)))
        else:
            base_premium = random.randint(750, 2500)
            cover_premium = sum(random.randint(50, 350) for cover in [
                'COMPREHENSIVE_COVER', 'COLLISION_COVER', 'UNINSURED_MOTORIST', 
                'MEDICAL_BENEFITS', 'PIP_COVER', 'ACCIDENT_FORGIVENESS', 
                'ROADSIDE_ASSISTANCE', 'ANTI_THEFT_DEVICE'
            ] if cover == 'YES')
            safety_score = random.uniform(0.15, 0.97) if random.random() > 0.78 else None
            safety_premium = safety_score * 1000 if safety_score is not None else 0
            umbrella_premium = random.choice([250, 500, 750, 1000])
            expected_premium = base_premium + cover_premium - safety_premium + umbrella_premium
            expected_premiums.append(max(expected_premium, 750))
    
    # Generate quote expected by date
    quote_expected_by_dates = [date + timedelta(days=random.randint(3, 10)) for date in application_dates]
    
    # Generate competition level
    competition_levels = random.choices(['Low Competition', 'Medium Competition', 'High Competition'], weights=[0.27, 0.57, 0.16], k=total_records)
    
    # Generate past three years no claims
    past_three_yrs_no_claims = random.choices(['0', '1', '2', '3 and above'], weights=[0.21, 0.17, 0.45, 0.17], k=total_records)
    
    # Generate past three years claim amount
    past_three_yrs_claim_amt = [random.randint(1000, 10000) if claims != '0' else 0 for claims in past_three_yrs_no_claims]
    
    # Generate number of drivers
    no_of_drivers = random.choices([1, 2, 3, 4], weights=[0.5, 0.3, 0.15, 0.05], k=total_records)
    
    # Generate coverages (continued)
    comprehensive_cover = random.choices(['YES', 'NO'], weights=[0.2389, 0.7611], k=total_records)
    collision_cover = random.choices(['YES', 'NO'], weights=[0.6721, 0.3279], k=total_records)
    uninsured_motorist = random.choices(['YES', 'NO'], weights=[0.2468, 0.7532], k=total_records)
    medical_benefits = random.choices(['YES', 'NO'], weights=[0.2389, 0.7611], k=total_records)
    pip_cover = random.choices(['YES', 'NO'], weights=[0.6721, 0.3279], k=total_records)
    accident_forgiveness = random.choices(['YES', 'NO'], weights=[0.2468, 0.7532], k=total_records)
    roadside_assistance = random.choices(['YES', 'NO'], weights=[0.7432, 0.2568], k=total_records)
    loss_of_use = random.choices(['YES', 'NO'], weights=[0.2913, 0.7087], k=total_records)
    anti_theft_device = random.choices(['YES', 'NO'], weights=[0.2913, 0.7087], k=total_records)
    
    # Generate safety score
    safety_score = [random.uniform(0.15, 0.97) if random.random() > 0.78 else None for _ in range(total_records)]
    
    # Generate quote date
    quote_dates = [date + timedelta(days=random.randint(0, 3)) for date in quote_expected_by_dates]
    
    # Generate quote acceptance date
    quote_acceptance_dates = [date + timedelta(days=random.randint(0, 10)) if converted == 'Quote Converted' else None for date, converted in zip(quote_dates, quote_converted)]
    
    # Ensure QUOTE_ACCEPTANCE_DATE is always lesser than POLICY_BIND_DATE
    quote_acceptance_dates = [date if date is None or date < datetime.strptime(bind_date, '%Y-%m-%d') else datetime.strptime(bind_date, '%Y-%m-%d') - timedelta(days=random.randint(1, 10)) for date, bind_date in zip(quote_acceptance_dates, policy_bind_dates)]
    
    # Generate loss reason
    loss_reason = [None if converted == 'Quote Converted' else random.choices(['Lower Premium', 'Lower Deductible', 'Higher CSL', 'Better Coverages'], weights=[0.32, 0.18, 0.27, 0.23])[0] for converted in quote_converted]
    
    # Check lengths of all lists
    lengths = [len(application_ids), len(application_dates), len(quote_converted), len(policy_numbers), len(customer_ids), len(policy_premiums), len(policy_deductibles), len(policy_csls), len(policy_bind_dates), len(agent_ids), len(prospect_ids), len(prospect_state), len(prospect_counties), len(prospect_cities), len(auto_makes), len(auto_models), len(expected_csls), len(deductible_looked_for), len(expected_premiums), len(quote_expected_by_dates), len(competition_levels), len(past_three_yrs_no_claims), len(past_three_yrs_claim_amt), len(no_of_drivers), len(comprehensive_cover), len(collision_cover), len(uninsured_motorist), len(medical_benefits), len(pip_cover), len(accident_forgiveness), len(roadside_assistance), len(loss_of_use), len(anti_theft_device), len(safety_score), len(quote_dates), len(quote_acceptance_dates), len(loss_reason)]
    
    if len(set(lengths)) != 1:
        raise ValueError(f"All arrays must be of the same length. Lengths: {lengths}")
    
    # Append the generated data to the synthetic data DataFrame
    year_data = pd.DataFrame({
        'APPLICATION_ID': application_ids,
        'APPLICATION_DATE': application_dates,
        'QUOTE_CONVERTED': quote_converted,
        'POLICY_NUMBER': policy_numbers,
        'CUSTOMER_ID': customer_ids,
        'POLICY_PREMIUM': policy_premiums,
        'POLICY_DEDUCTIBLE': policy_deductibles,
        'POLICY_CSL': policy_csls,
        'POLICY_BIND_DATE': policy_bind_dates,
        'AGENT_ID': agent_ids,
        'PROSPECT_ID': prospect_ids,
        'PROSPECT_STATE': prospect_state,
        'PROSPECT_COUNTY': prospect_counties,
        'PROSPECT_CITY': prospect_cities,
        'AUTO_MAKE': auto_makes,
        'AUTO_MODEL': auto_models,
        'EXPECTED_CSL': expected_csls,
        'DEDUCTIBLE_LOOKED_FOR': deductible_looked_for,
        'EXPECTED_PREMIUM': expected_premiums,
        'QUOTE_EXPECTED_BY_DATE': quote_expected_by_dates,
        'COMPETITION_LEVEL': competition_levels,
        'PAST_THREE_YRS_NO_CLAIMS': past_three_yrs_no_claims,
        'PAST_THREE_YRS_CLAIM_AMT': past_three_yrs_claim_amt,
        'NO_OF_DRIVERS': no_of_drivers,
        'COMPREHENSIVE_COVER': comprehensive_cover,
        'COLLISION_COVER': collision_cover,
        'UNINSURED_MOTORIST': uninsured_motorist,
        'MEDICAL_BENEFITS': medical_benefits,
        'PIP_COVER': pip_cover,
        'ACCIDENT_FORGIVENESS': accident_forgiveness,
        'ROADSIDE_ASSISTANCE': roadside_assistance,
        'LOSS_OF_USE': loss_of_use,
        'ANTI_THEFT_DEVICE': anti_theft_device,
        'SAFETY_SCORE': safety_score,
        'QUOTE_DATE': quote_dates,
        'QUOTE_ACCEPTANCE_DATE': quote_acceptance_dates,
        'LOSS_REASON': loss_reason
    })
    
    synthetic_data = pd.concat([synthetic_data, year_data], ignore_index=True)

# Save the synthetic data to the specified file path
synthetic_data.to_csv('C:/Users/10741867/OneDrive - LTIMindtree/Documents/Datasets/Solution Datasets/Solution 1 - PNC Quote Conversion and Prioritization/Datasets/Regenerated Datasets/AUTO_INSURANCE_QUOTE_CONVERSION_MASTER.csv', index=False)

Generating data: 100%|██████████| 10/10 [02:00<00:00, 12.01s/it]


In [22]:
synthetic_data.shape

(2577082, 37)

In [23]:
synthetic_data.head()

,APPLICATION_ID,APPLICATION_DATE,QUOTE_CONVERTED,POLICY_NUMBER,CUSTOMER_ID,POLICY_PREMIUM,POLICY_DEDUCTIBLE,POLICY_CSL,POLICY_BIND_DATE,AGENT_ID,...,MEDICAL_BENEFITS,PIP_COVER,ACCIDENT_FORGIVENESS,ROADSIDE_ASSISTANCE,LOSS_OF_USE,ANTI_THEFT_DEVICE,SAFETY_SCORE,QUOTE_DATE,QUOTE_ACCEPTANCE_DATE,LOSS_REASON
0,APP00000000,2021-05-29,Quote Converted,P71658449,CUST800747,3999.0,250.0,250/500,2018-02-16,AG716533,...,NO,YES,YES,NO,NO,NO,NaN,2021-06-04,2018-02-06,None
1,APP00000001,2011-07-07,Quote Converted,P71757070,CUST779285,2165.0,500.0,250/500,2017-10-10,AG716621,...,YES,NO,NO,YES,NO,YES,NaN,2011-07-11,2011-07-21,None
2,APP00000002,2017-01-27,Quote Converted,P71681128,CUST768419,1567.0,500.0,100/300,2021-02-22,AG716702,...,NO,YES,NO,YES,YES,NO,0.814261,2017-02-06,2017-02-15,None
3,APP00000003,2006-10-03,Quote Converted,P71732892,CUST724882,2511.0,1000.0,100/300,2017-05-22,AG716426,...,YES,NO,NO,YES,YES,YES,0.455145,2006-10-09,2006-10-10,None
4,APP00000004,2015-10-13,Quote Converted,P71672682,CUST806386,2764.0,250.0,250/500,2019-07-02,AG716344,...,NO,YES,NO,NO,NO,NO,NaN,2015-10-21,2015-10-26,None


In [24]:
# Filter the rows where POLICY_NUMBER is null
filtered_data = synthetic_data[synthetic_data['POLICY_NUMBER'].isnull()]

# Display the number of rows
num_rows = filtered_data.shape[0]
print(f"Number of rows where POLICY_NUMBER is null: {num_rows}")

# Display the head of the filtered data
filtered_data.head()

Number of rows where POLICY_NUMBER is null: 1623560


,APPLICATION_ID,APPLICATION_DATE,QUOTE_CONVERTED,POLICY_NUMBER,CUSTOMER_ID,POLICY_PREMIUM,POLICY_DEDUCTIBLE,POLICY_CSL,POLICY_BIND_DATE,AGENT_ID,...,MEDICAL_BENEFITS,PIP_COVER,ACCIDENT_FORGIVENESS,ROADSIDE_ASSISTANCE,LOSS_OF_USE,ANTI_THEFT_DEVICE,SAFETY_SCORE,QUOTE_DATE,QUOTE_ACCEPTANCE_DATE,LOSS_REASON
55011,APP00055011,2017-09-19,Quote not converted,None,None,NaN,NaN,None,None,None,...,YES,NO,NO,YES,NO,YES,0.443211,2017-09-24,NaT,Higher CSL
55012,APP00055012,2018-11-14,Quote not converted,None,None,NaN,NaN,None,None,None,...,YES,YES,NO,YES,NO,NO,0.456556,2018-11-24,NaT,Lower Premium
55013,APP00055013,2010-09-10,Quote not converted,None,None,NaN,NaN,None,None,None,...,NO,YES,NO,NO,NO,YES,0.351509,2010-09-20,NaT,Lower Premium
55014,APP00055014,2019-01-15,Quote not converted,None,None,NaN,NaN,None,None,None,...,NO,NO,NO,YES,YES,YES,0.599585,2019-01-27,NaT,Higher CSL
55015,APP00055015,2018-08-07,Quote not converted,None,None,NaN,NaN,None,None,None,...,YES,YES,YES,NO,YES,NO,NaN,2018-08-18,NaT,Lower Deductible


In [25]:
# Filter the rows where POLICY_NUMBER is NOT null
filtered_data_not_null = synthetic_data[synthetic_data['POLICY_NUMBER'].notnull()]

# Display the number of rows
num_rows_not_null = filtered_data_not_null.shape[0]
print(f"Number of rows where POLICY_NUMBER is NOT null: {num_rows_not_null}")

# Display the head of the filtered data
filtered_data_not_null.head()

Number of rows where POLICY_NUMBER is NOT null: 953522


,APPLICATION_ID,APPLICATION_DATE,QUOTE_CONVERTED,POLICY_NUMBER,CUSTOMER_ID,POLICY_PREMIUM,POLICY_DEDUCTIBLE,POLICY_CSL,POLICY_BIND_DATE,AGENT_ID,...,MEDICAL_BENEFITS,PIP_COVER,ACCIDENT_FORGIVENESS,ROADSIDE_ASSISTANCE,LOSS_OF_USE,ANTI_THEFT_DEVICE,SAFETY_SCORE,QUOTE_DATE,QUOTE_ACCEPTANCE_DATE,LOSS_REASON
0,APP00000000,2021-05-29,Quote Converted,P71658449,CUST800747,3999.0,250.0,250/500,2018-02-16,AG716533,...,NO,YES,YES,NO,NO,NO,NaN,2021-06-04,2018-02-06,None
1,APP00000001,2011-07-07,Quote Converted,P71757070,CUST779285,2165.0,500.0,250/500,2017-10-10,AG716621,...,YES,NO,NO,YES,NO,YES,NaN,2011-07-11,2011-07-21,None
2,APP00000002,2017-01-27,Quote Converted,P71681128,CUST768419,1567.0,500.0,100/300,2021-02-22,AG716702,...,NO,YES,NO,YES,YES,NO,0.814261,2017-02-06,2017-02-15,None
3,APP00000003,2006-10-03,Quote Converted,P71732892,CUST724882,2511.0,1000.0,100/300,2017-05-22,AG716426,...,YES,NO,NO,YES,YES,YES,0.455145,2006-10-09,2006-10-10,None
4,APP00000004,2015-10-13,Quote Converted,P71672682,CUST806386,2764.0,250.0,250/500,2019-07-02,AG716344,...,NO,YES,NO,NO,NO,NO,NaN,2015-10-21,2015-10-26,None
